In [3]:
from yaml import safe_load

with open('.client_config.yaml') as file:
    configs: dict[str, dict[str]] = safe_load(file)

from os import getenv
API: str = str(getenv('API'))
config = configs['semic']['init']
from openai import OpenAI, AzureOpenAI
client = AzureOpenAI(**config)


In [2]:
with open("chat_history/system_prompt.txt") as f:
    print(f.read())

# TASK
You are an agent, expert in semantic modelling.
You help the user by:
- Providing information about concepts (classes, attributes, ...) that they do not know.
- Suggest existing concepts that the user can reuse instead of creating new ones.
- Generate a data model fitting a user's given description, using the user's given guidelines.

# PERSISTENCE
Keep going until the userâ€™s query is completely resolved, before ending your turn and yielding back to the user. Only terminate your turn when you are sure that the problem is solved, or if you think there are not information available to you to solve the problem.

# PLANNING
You MUST plan extensively before each function/tool call, and reflect extensively on the outcomes of the previous function calls. DO NOT do this entire process by making function calls only, as this can impair your ability to solve the problem and think insightfully.

# TOOLS

** 1. SEARCH
If you do not have sufficient information to answer the user's query, re

In [1]:
a = {'1': 2}
b = {'1': 3, '3': 1}
b |= a
b

{'1': 2, '3': 1}

In [ ]:
API, configs

('sharedservice',
 {'shared_service': {'model': 'azure.gpt-4o',
   'base_url': 'https://genai-sharedservice-emea.pwc.com',
   'api_key': 'sk-9SCcnIyb_E475qTiG0lv7A'},
  'semic': {'model': 'gpt-4',
   'base_url': '= https://ecdigitb2semic9142630917.openai.azure.com/depolyments/gpt-4',
   'api_key': '10764a495ad64953b8ca4bf3b0bc7872'}})

In [7]:
from io import BytesIO
from pathlib import Path
from pprint import pprint

# Read JSON-LD file path into BytesIO
def read_jsonld_file(file_path: Path) -> str:
    with open(file_path, 'rb') as f:
        bytes_data: BytesIO = BytesIO(f.read())
        #bytes_data.seek(0)
        string_data: str = bytes_data.getvalue().decode('utf-8')
    return string_data


In [2]:
file_path: Path = Path('test.jsonld')
json_ld_data: str = read_jsonld_file(file_path)
print(json_ld_data)


{
    "@context": {
      "foaf": "http://xmlns.com/foaf/0.1/",
      "schema": "http://schema.org/",
      "vcard": "http://www.w3.org/2006/vcard/ns#",
      "xsd": "http://www.w3.org/2001/XMLSchema#"
    },
    "@type": "foaf:Person",
    "foaf:name": "Lucas",
    "foaf:gender": "male",
    "schema:nationality": {
      "@type": "schema:Country",
      "schema:name": "Belgium"
    },
    "schema:address": {
      "@type": "vcard:Address",
      "vcard:street-address": "Dorpstraat 17",
      "vcard:locality": "Nieuwpoort",
      "vcard:region": "West-Flanders",
      "vcard:country-name": "Belgium"
    },
    "schema:identifier": {
      "@type": "schema:PropertyValue",
      "schema:propertyID": "nationalRegistryNumber",
      "schema:value": "Your National Registry Number Here"
    }
}


In [9]:
prompt: str = """system:
You are an AI assistant that exists to use the user's jsonld context scheme, and to get the namespace in a certain form.

assistant:
The user's context scheme:"""+json_ld_data+"""\nassistant:
context_scheme

The form of the namespaces:
{
"name":"http://www.namespace.com/ns/",
"tree":"http://www.forest.com/voc/tree#"
}"""
print(prompt)

system:
You are an AI assistant that exists to use the user's jsonld context scheme, and to get the namespace in a certain form.

assistant:
The user's context scheme:{
    "@context": {
      "foaf": "http://xmlns.com/foaf/0.1/",
      "schema": "http://schema.org/",
      "vcard": "http://www.w3.org/2006/vcard/ns#",
      "xsd": "http://www.w3.org/2001/XMLSchema#"
    },
    "@type": "foaf:Person",
    "foaf:name": "Lucas",
    "foaf:gender": "male",
    "schema:nationality": {
      "@type": "schema:Country",
      "schema:name": "Belgium"
    },
    "schema:address": {
      "@type": "vcard:Address",
      "vcard:street-address": "Dorpstraat 17",
      "vcard:locality": "Nieuwpoort",
      "vcard:region": "West-Flanders",
      "vcard:country-name": "Belgium"
    },
    "schema:identifier": {
      "@type": "schema:PropertyValue",
      "schema:propertyID": "nationalRegistryNumber",
      "schema:value": "Your National Registry Number Here"
    }
}
assistant:
context_scheme

The fo

get namespaces

In [13]:
from utils.sk_setup import initialize_kernel
import semantic_kernel as sk
from semantic_kernel import (
    Kernel
)

kernel: Kernel = initialize_kernel()
async def chat_with_jsonld(kernel: Kernel, json_ld_data: str):
    plugins_directory: Path = Path("./plugins")
    function_list = kernel.import_plugin_from_prompt_directory(
        plugins_directory, 
        "JsonldPlugin"
    )
    chat_function = function_list["GetContextUrls"]
    
    response = str(await kernel.invoke(chat_function, sk.KernelArguments(context_scheme=json_ld_data)))

    return response

In [15]:
import json

def gpt_ouput_to_json(gpt_output):
    try:
        structured_output = json.loads(gpt_output)
    except json.JSONDecodeError:
        structured_output = json.loads(gpt_output[7:-3])
    return structured_output

In [16]:
from pprint import pprint
import json

response = str(await chat_with_jsonld(kernel, json_ld_data))

namespaces = gpt_ouput_to_json(response)

#response = response.replace("\n", "")

AttributeError: 'Kernel' object has no attribute 'import_plugin_from_prompt_directory'

In [62]:
namespaces

{'Address': 'http://www.w3.org/ns/locn#',
 'Agent': 'http://xmlns.com/foaf/0.1/',
 'Code': 'http://www.w3.org/2004/02/skos/core#',
 'ContactPoint': 'http://data.europa.eu/m8g/',
 'Date': 'http://www.w3.org/2001/XMLSchema#',
 'Document': 'http://xmlns.com/foaf/0.1/',
 'GenericDate': 'http://data.europa.eu/m8g/',
 'Identifier': 'http://www.w3.org/ns/adms#',
 'Jurisdiction': 'http://purl.org/dc/terms/',
 'Literal': 'http://www.w3.org/2000/01/rdf-schema#',
 'Location': 'http://purl.org/dc/terms/',
 'Person': 'http://www.w3.org/ns/person#',
 'Text': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
 'Uri': 'http://www.w3.org/2001/XMLSchema#'}

In [48]:
json_ld_instance = """{  "@context": ["""+str(response)+"""}],"""

In [49]:
json_ld_instance

'{  "@context": [{\'Address\': \'http://www.w3.org/ns/locn#\', \'Agent\': \'http://xmlns.com/foaf/0.1/\', \'ContactPoint\': \'http://data.europa.eu/m8g/\', \'GenericDate\': \'http://data.europa.eu/m8g/\', \'Identifier\': \'http://www.w3.org/ns/adms#\', \'Jurisdiction\': \'http://purl.org/dc/terms/\', \'Location\': \'http://purl.org/dc/terms/\', \'Person\': \'http://www.w3.org/ns/person#\'}}],'

get graph

In [51]:
user_prompt = "I want to transform the following real-life example into a JSON-LD formatted data example based on the Core Person Vocabulary: Lucas is a male Belgian citizen that lives in West-Flanders. He has a national registry number and his address is Dorpstraat 17 Nieuwpoort."

In [83]:
import semantic_kernel as sk

async def create_graph(kernel, user_prompt, json_ld_data, namespaces):
    plugins_directory = "./plugins"
    function_list = kernel.import_plugin_from_prompt_directory(plugins_directory, "JsonldPlugin")
    chat_function = function_list["CreateGraphPart"]
    print(f"user_prompt:\n{user_prompt}\n\njson_ld_data:\n{json_ld_data}\n\nnamespaces:\n{namespaces}")
    response = str(await kernel.invoke(chat_function, sk.KernelArguments(user_prompt=user_prompt, namespaces=json.dumps(namespaces), context_scheme=json_ld_data)))

    return response

In [81]:
json_ld_data

'{\n  "@context": {\n    "Address": "http://www.w3.org/ns/locn#Address",\n    "Address.addressArea": {\n      "@container": "@set",\n      "@id": "http://www.w3.org/ns/locn#addressArea",\n      "@type": "http://www.w3.org/1999/02/22-rdf-syntax-ns#langString"\n    },\n    "Address.addressId": {\n      "@container": "@set",\n      "@id": "http://www.w3.org/ns/locn#addressId"\n    },\n    "Address.administrativeUnitLevel1": {\n      "@container": "@set",\n      "@id": "http://www.w3.org/ns/locn#adminUnitL1",\n      "@type": "http://www.w3.org/1999/02/22-rdf-syntax-ns#langString"\n    },\n    "Address.administrativeUnitLevel2": {\n      "@container": "@set",\n      "@id": "http://www.w3.org/ns/locn#adminUnitL2",\n      "@type": "http://www.w3.org/1999/02/22-rdf-syntax-ns#langString"\n    },\n    "Address.fullAddress": {\n      "@container": "@set",\n      "@id": "http://www.w3.org/ns/locn#fullAddress",\n      "@type": "http://www.w3.org/1999/02/22-rdf-syntax-ns#langString"\n    },\n    "Ad

In [80]:
namespaces

{'Address': 'http://www.w3.org/ns/locn#',
 'Agent': 'http://xmlns.com/foaf/0.1/',
 'Code': 'http://www.w3.org/2004/02/skos/core#',
 'ContactPoint': 'http://data.europa.eu/m8g/',
 'Date': 'http://www.w3.org/2001/XMLSchema#',
 'Document': 'http://xmlns.com/foaf/0.1/',
 'GenericDate': 'http://data.europa.eu/m8g/',
 'Identifier': 'http://www.w3.org/ns/adms#',
 'Jurisdiction': 'http://purl.org/dc/terms/',
 'Literal': 'http://www.w3.org/2000/01/rdf-schema#',
 'Location': 'http://purl.org/dc/terms/',
 'Person': 'http://www.w3.org/ns/person#',
 'Text': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
 'Uri': 'http://www.w3.org/2001/XMLSchema#'}

In [84]:
graph = await create_graph(kernel, user_prompt, json_ld_instance, namespaces)
graph = gpt_ouput_to_json(graph)
graph

Overwriting function "ChatWithJsonld" in collection
Overwriting function "CreateGraphPart" in collection
Overwriting function "GetContextUrls" in collection


I want to transform the following real-life example into a JSON-LD formatted data example based on the Core Person Vocabulary: Lucas is a male Belgian citizen that lives in West-Flanders. He has a national registry number and his address is Dorpstraat 17 Nieuwpoort. {  "@context": [{'Address': 'http://www.w3.org/ns/locn#', 'Agent': 'http://xmlns.com/foaf/0.1/', 'ContactPoint': 'http://data.europa.eu/m8g/', 'GenericDate': 'http://data.europa.eu/m8g/', 'Identifier': 'http://www.w3.org/ns/adms#', 'Jurisdiction': 'http://purl.org/dc/terms/', 'Location': 'http://purl.org/dc/terms/', 'Person': 'http://www.w3.org/ns/person#'}}], {'Address': 'http://www.w3.org/ns/locn#', 'Agent': 'http://xmlns.com/foaf/0.1/', 'Code': 'http://www.w3.org/2004/02/skos/core#', 'ContactPoint': 'http://data.europa.eu/m8g/', 'Date': 'http://www.w3.org/2001/XMLSchema#', 'Document': 'http://xmlns.com/foaf/0.1/', 'GenericDate': 'http://data.europa.eu/m8g/', 'Identifier': 'http://www.w3.org/ns/adms#', 'Jurisdiction': 'ht

{'@graph': [{'@id': 'ex:person',
   '@type': 'Person',
   'Person.birthName': {'@language': 'en', '@value': 'John Doe'},
   'Person.dateOfBirth': {'@type': 'xsd:date', '@value': '1980-01-01'},
   'Person.dateOfDeath': {'@type': 'xsd:date', '@value': '2050-12-31'},
   'Person.domicile': {'@id': 'ex:address'},
   'Person.sex': {'@id': 'cv:Male'}}]}

contact semic index api

In [13]:
AZURE_OPENAI_API_KEY="10764a495ad64953b8ca4bf3b0bc7872"
AZURE_OPENAI_ENDPOINT="https://ecdigitb2semic9142630917.openai.azure.com/"
AZURE_SEARCH_SERVICE_ENDPOINT="https://ai-search-semic-poc.search.windows.net"
AZURE_SEARCH_INDEX_NAME="cpsv-ap-v01"
AZURE_SEARCH_ADMIN_KEY="nMt4yalOISlTDaj4eSbrI5jM7sVO8J9Op4i3FbNlWFAzSeC2FNur"
AZURE_COGNITIVE_SEARCH_SERVICE_NAME="ai-search-semic-poc"
AZURE_COGNITIVE_SEARCH_API_KEY="nMt4yalOISlTDaj4eSbrI5jM7sVO8J9Op4i3FbNlWFAzSeC2FNur"
OPENAI_API_VERSION="2023-06-01-preview"
AZURE_SEARCH_INDEX_NAME2="eurovoc-v01"
AZURE_SEARCH_SEMANTIC_SEARCH_CONFIG="semic-poc-semantic-config"
AZURE_SEARCH_INDEX_IS_PRECHUNKED=False
AZURE_SEARCH_TOP_K=5
AZURE_SEARCH_ENABLE_IN_DOMAIN=True
AZURE_SEARCH_QUERY_TYPE="semantic"
AZURE_SEARCH_STRICTNESS=3
AZURE_OPENAI_SYSTEM_MESSAGE="You are an AI assistant that helps data modelers find information on concepts, relations, properties, ..."

In [15]:
import requests

# Define the endpoint and API key
base_url = AZURE_SEARCH_SERVICE_ENDPOINT
api_version = "2023-11-01"
index_name = AZURE_SEARCH_INDEX_NAME
api_key = AZURE_SEARCH_ADMIN_KEY

# Construct the URL
url = f"{base_url}/indexes/{index_name}/docs/search?api-version={api_version}"

# Define the headers
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# Define the query
query = {
    "search": "medicalHistory",
    #"select": "HotelId, HotelName, Tags, Description",
    #"searchFields": "Description, Tags",
    "count": True
}

# Make the POST request
response = requests.post(url, headers=headers, json=query)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON
    results = response.json()
    from pprint import pprint
    pprint(results)
else:
    print(f"Error: {response.status_code}")
    print(response.text)


{'@odata.context': "https://ai-search-semic-poc.search.windows.net/indexes('cpsv-ap-v01')/$metadata#docs(*)",
 '@odata.count': 0,
 'value': []}


In [12]:
[{
    "link": results["value"][0]["Title"],
    "definition": results["value"][0]["Content"]
},{
    "link": results["value"][1]["Title"],
    "definition": results["value"][1]["Content"]
},{
    "link": results["value"][2]["Title"],
    "definition": results["value"][2]["Content"]
},{
    "link": results["value"][3]["Title"],
    "definition": results["value"][3]["Content"]
},{
    "link": results["value"][4]["Title"],
    "definition": results["value"][4]["Content"]
}]

[{'link': 'LOV: http://semanticscience.org/resource/SIO_010673',
  'definition': '\n                medical history\n                A medical history is a record of the events of a recipient of medical care.\n                '},
 {'link': 'LOV: http://www.w3.org/2001/sw/hcls/ns/transmed/TMO_0020',
  'definition': "\n                personal medical history\n                A personal medical history is a medical record consisting of a collection of information obtained from the patient and from other sources concerning the patient's health.\n                "},
 {'link': 'LOV: http://semanticscience.org/resource/SIO_001027',
  'definition': "\n                medical health record\n                A medical health record is a record of a single patient's medical history.\n                "},
 {'link': 'LOV: https://w3id.org/skgo/modsci#History',
  'definition': '\n                History of Science\n                The history of science is the study of the development of science and 

In [7]:
import re

def extract_words(line):
    # Gebruik regex om de gewenste patronen te matchen
    pattern = r'k=[^:]*::([^|]+)|r=([^|]+)'
    matches = re.findall(pattern, line)
    
    # Haal de juiste groepen op en verwijder lege waarden
    words = [word for match in matches for word in match if word]
    
    return words

# Voorbeeldgebruik
line = "k=CCCEV::Bewijs|a=taal"
result = extract_words(line)
print(result)


['Bewijs']


In [17]:
from api_calls.search_api_call import search_index

results = search_index("public service")

In [18]:
results

{'@odata.context': "https://ai-search-semic-poc.search.windows.net/indexes('cpsv-ap-v01')/$metadata#docs(*)",
 '@odata.count': 2377,
 '@search.nextPageParameters': {'search': 'public service',
  'count': True,
  'skip': 50},
 'value': [{'@search.score': 47.412987,
   'id': 'LOV_49747',
   'Title': 'LOV: http://vocab.org/transit/terms/Service',
   'Keyword': '<http://vocab.org/transit/terms/Service> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#Class> .\n<http://vocab.org/transit/terms/Service> <http://www.w3.org/2003/06/sw-vocab-status/ns#term_status> "unstable" .\n<http://vocab.org/transit/terms/Service> <http://www.w3.org/2000/01/rdf-schema#label> "Service"@en .\n<http://vocab.org/transit/terms/Service> <http://www.w3.org/2000/01/rdf-schema#isDefinedBy> <http://vocab.org/transit/terms/> .\n<http://vocab.org/transit/terms/Service> <http://purl.org/dc/terms/issued> "2011-03-28"^^<http://www.w3.org/2001/XMLSchema#date> .\n<http://vocab.org/transit/term

In [64]:
values = [result["Keyword"].split('\n') for result in results["value"]]

In [65]:
found_definitions = []
for i in range(len(values)):
    for j in range(len(values[i])):
        triple_split = values[i][j].split(" ")
        if len(values[i][j]) > 3:
                triple_list = [triple_split[0], triple_split[1], " ".join(triple_split[2:])]
                if "#comment" in triple_list[1]:
                    found_definitions.append(triple_list[2])

In [66]:
found_definitions

['"A public transport service that operates a route on a given schedule."@en .',
 '"Status of the public service process."@en .',
 '"Result of the public service process."@en .',
 '"This class represents the service itself. A public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a public agency for the benefit of a citizen, a business or another public agency."@en .',
 '"This class represents the service itself. As noted in the scope (section 1.4), a public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a public agency for the benefit of a citizen, a business or another public agency."@en .',
 '"This class represents the service itself. A public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a 

In [61]:
for i in range(len(values)):
    for j in range(len(values[i])):
        if "#comment" in values[i][j][1]:
            print(values[i][j][2])

"A public transport service that operates a route on a given schedule."@en .
"Status of the public service process."@en .
"Result of the public service process."@en .
"This class represents the service itself. A public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a public agency for the benefit of a citizen, a business or another public agency."@en .
"This class represents the service itself. As noted in the scope (section 1.4), a public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a public agency for the benefit of a citizen, a business or another public agency."@en .
"This class represents the service itself. A public service is the capacity to carry out a procedure and exists whether it is used or not. It is a set of deeds and acts performed by or on behalf of a public agency for the 

In [50]:
values

[[['<http://vocab.org/transit/terms/Service>',
   '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
   '<http://www.w3.org/2002/07/owl#Class> .'],
  ['<http://vocab.org/transit/terms/Service>',
   '<http://www.w3.org/2003/06/sw-vocab-status/ns#term_status>',
   '"unstable" .'],
  ['<http://vocab.org/transit/terms/Service>',
   '<http://www.w3.org/2000/01/rdf-schema#label>',
   '"Service"@en .'],
  ['<http://vocab.org/transit/terms/Service>',
   '<http://www.w3.org/2000/01/rdf-schema#isDefinedBy>',
   '<http://vocab.org/transit/terms/> .'],
  ['<http://vocab.org/transit/terms/Service>',
   '<http://purl.org/dc/terms/issued>',
   '"2011-03-28"^^<http://www.w3.org/2001/XMLSchema#date> .'],
  ['<http://vocab.org/transit/terms/Service>',
   '<http://www.w3.org/2000/01/rdf-schema#comment>',
   '"A public transport service that operates a route on a given schedule."@en .'],
  ''],
 [['<http://purl.org/oslo/ns/localgov#status>',
   '<http://www.w3.org/2000/01/rdf-schema#domain>',
   '<http:

In [3]:
str(["1", "2", "3", "4", "5"])

"['1', '2', '3', '4', '5']"

In [1]:
from json import (
    load,
    dump,
)

with open(r"C:\Users\ecaudron001\Documents\GitHub\AI4Semantics-MCP-Server\resources\semantic_model\models\Emilien\iop.json", "r") as f:
    model = load(f)

In [21]:
from tabs.xmi_tab.scripts_xmi_chat.model_utils.chat_data_structure import shorten_json, _shorten_elements, _shorten_class

In [28]:
def _shorten_class(elem):
    class_dict = {}
    class_dict["name"] = elem["name"]

    tags = []
    for tag in elem["tags"]:
        if "definition" in tag["name"] or "uri" in tag["name"]:
            tag_dict = {}
            tag_dict["name"] = tag["name"]
            tag_dict["value"] = tag["value"]
            tags.append(tag_dict)
    class_dict["tags"] = tags

    try:
        attributes = []
        for attribute in elem["attributes"]:
            attribute_dict = {}
            attribute_dict["name"] = attribute["name"]
            attribute_dict["type"] = attribute["type"]
            #attribute_dict["lower_bounds"] = attribute["lower_bounds"]
            #attribute_dict["upper_bounds"] = attribute["upper_bounds"]
            attribute_dict["tags"] = attribute["tags_attribute"]
            attributes.append(attribute_dict)

        class_dict["attributes"] = attributes

    except Exception:
        pass

    return class_dict

In [29]:
_shorten_class(model["elements"][2])

{'name': 'ActionPlan',
 'tags': [{'name': 'definition-en', 'value': 'An action plan'},
  {'name': 'uri',
   'value': 'http://data4wallonia.com/resource/model/ActionPlan'}],
 'attributes': [{'name': 'additionalDescription',
   'type': 'Text',
   'tags': [{'name': 'uri',
     'value': 'http://data4wallonia.com/resource/model/additionalDescription'},
    {'name': 'definition-en', 'value': 'An additional description'},
    {'name': 'label-en', 'value': 'additional description'}]},
  {'name': 'category',
   'type': 'Category',
   'tags': [{'name': 'uri',
     'value': 'http://data4wallonia.com/resource/model/category'},
    {'name': 'definition-en', 'value': 'A classification'},
    {'name': 'label-en', 'value': 'category'}]},
  {'name': 'clientSite',
   'type': 'String',
   'tags': [{'name': 'uri',
     'value': 'http://data4wallonia.com/resource/model/clientSite'},
    {'name': 'definition-en', 'value': 'A client site'},
    {'name': 'label-en', 'value': 'client site'}]},
  {'name': 'emai

In [16]:
for attribute in model["elements"][2]["attributes"]:
    print(attribute)
    break

{'name': 'additionalDescription', 'type': 'Text', 'lower_bounds': '0', 'upper_bounds': '1', 'tags_attribute': [{'name': 'uri', 'value': 'http://data4wallonia.com/resource/model/additionalDescription'}, {'name': 'definition-en', 'value': 'An additional description'}, {'name': 'label-en', 'value': 'additional description'}]}
